In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, log_loss
from catboost import CatBoostClassifier

def pre_df(df):
    df = df.drop(["id"], axis=1)

    # df['Classic angina'] = ((df['Chest pain type'] == 4) & (df['Exercise angina'] == 1)).astype(int)
    # df['High risk'] = (((df['Thallium'] == 7) | (df['Thallium'] == 6)) & (df['Number of vessels fluro'] >= 1) & (df['ST depression'] > 1)).astype(int)

    # df['HR deficit'] = (df['Max HR'] + df['Age'] < 200).astype(int)

    # df['Framingham risk'] = ((df['Age'] > 55) * 3 + df['Sex'] * 2 + (df['BP'] > 140) * 2 + (df['Cholesterol'] > 240) * 2 + df['FBS over 120'])
    # df['Ischemic burden'] = (df['Exercise angina'] * 10 + df['ST depression'] * 5 + (3 - df['Slope of ST']) * 3)
    # df['Vessel severity'] = (df['Number of vessels fluro'] * 5 + (df['Thallium'] == 6) * 3 + (df['Thallium'] == 7) * 8)

    # df['Thallium chest'] = df['Thallium'] * df['Chest pain type']
    # df['Age sex chest'] = df['Age'] * df['Sex'] * df['Chest pain type']
    # df['Thallium vessels ST'] = df['Thallium'] * df['Number of vessels fluro'] * df['ST depression']
    # df['Health'] = df['Max HR'] / 2 - df['ST depression'] * 10 - df['Exercise angina'] * 15 - df['Number of vessels fluro'] * 10
    return df

TRAIN_PATH = "../data/train.csv"
TEST_PATH = "../data/test.csv"
SUB_PATH = "../data/sample_submission.csv"

train = pre_df(pd.read_csv(TRAIN_PATH))
test = pre_df(pd.read_csv(TEST_PATH))

train["Target"] = (train["Heart Disease"] == "Presence") * 1
train = train.drop(["Heart Disease"], axis=1)

TARGET_COL = train.columns[-1]
FEATURE_COLS = list(train.columns)[:-1]
X = train[FEATURE_COLS].copy()
y = train[TARGET_COL].copy()
X_test = test[FEATURE_COLS].copy()

CAT_COLS = ["Sex", "Chest pain type", "FBS over 120", "EKG results", "Exercise angina", "Slope of ST", "Number of vessels fluro", "Thallium"] #\
# + ["Classic angina", "High risk", "HR deficit", "Thallium chest", "Age sex chest", "Thallium vessels ST"]
for c in CAT_COLS:
    X[c] = X[c].astype("string")
    X_test[c] = X_test[c].astype("string")
cat_features = [X.columns.get_loc(c) for c in CAT_COLS]



In [ ]:
N_SPLITS = 5
RANDOM_STATE = 42
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

test_pred = np.zeros(len(X_test))
score = 0

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",
        iterations=10000,
        learning_rate=0.03,
        depth=3,
        l2_leaf_reg=5.0,
        random_strength=1.0,
        bagging_temperature=1.0,
        border_count=128,
        random_seed=RANDOM_STATE + fold,
        verbose=200,
        task_type="GPU",
    )

    model.fit(
        X_train, y_train,
        cat_features=cat_features,
        eval_set=(X_valid, y_valid),
        use_best_model=True,
        early_stopping_rounds=200
    )

    test_pred += model.predict_proba(X_test)[:, 1] / N_SPLITS
    score += model.get_best_score()["validation"]["AUC"] / N_SPLITS

print(f"AUC: {score:.5f}")

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8878817	best: 0.8878817 (0)	total: 18.7ms	remaining: 3m 6s
200:	test: 0.9539225	best: 0.9539225 (200)	total: 3.4s	remaining: 2m 45s
400:	test: 0.9546245	best: 0.9546245 (400)	total: 8.25s	remaining: 3m 17s
600:	test: 0.9548950	best: 0.9548950 (600)	total: 15.5s	remaining: 4m 2s
800:	test: 0.9550980	best: 0.9550980 (800)	total: 23.7s	remaining: 4m 31s
1000:	test: 0.9552671	best: 0.9552672 (999)	total: 31.4s	remaining: 4m 42s
1200:	test: 0.9554009	best: 0.9554009 (1200)	total: 39.4s	remaining: 4m 48s
1400:	test: 0.9555038	best: 0.9555038 (1400)	total: 46.7s	remaining: 4m 46s
1600:	test: 0.9555872	best: 0.9555872 (1600)	total: 54.6s	remaining: 4m 46s
1800:	test: 0.9556611	best: 0.9556611 (1796)	total: 1m 2s	remaining: 4m 43s
2000:	test: 0.9557116	best: 0.9557124 (1995)	total: 1m 9s	remaining: 4m 37s
2200:	test: 0.9557573	best: 0.9557573 (2200)	total: 1m 15s	remaining: 4m 28s
2400:	test: 0.9557893	best: 0.9557896 (2399)	total: 1m 23s	remaining: 4m 23s
2600:	test: 0.9558211	best:

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8845354	best: 0.8845354 (0)	total: 34.3ms	remaining: 5m 43s
200:	test: 0.9528129	best: 0.9528129 (200)	total: 5.12s	remaining: 4m 9s
400:	test: 0.9535400	best: 0.9535400 (400)	total: 9.46s	remaining: 3m 46s
600:	test: 0.9538350	best: 0.9538350 (600)	total: 14.2s	remaining: 3m 42s
800:	test: 0.9540494	best: 0.9540494 (800)	total: 19.2s	remaining: 3m 40s
1000:	test: 0.9542458	best: 0.9542458 (1000)	total: 24.6s	remaining: 3m 40s
1200:	test: 0.9543709	best: 0.9543710 (1199)	total: 29.8s	remaining: 3m 38s
1400:	test: 0.9544725	best: 0.9544725 (1400)	total: 36.1s	remaining: 3m 41s
1600:	test: 0.9545500	best: 0.9545500 (1597)	total: 41.1s	remaining: 3m 35s
1800:	test: 0.9546119	best: 0.9546119 (1799)	total: 46.3s	remaining: 3m 30s
2000:	test: 0.9546649	best: 0.9546649 (2000)	total: 51.7s	remaining: 3m 26s
2200:	test: 0.9547071	best: 0.9547072 (2198)	total: 57.2s	remaining: 3m 22s
2400:	test: 0.9547344	best: 0.9547346 (2392)	total: 1m 3s	remaining: 3m 20s
2600:	test: 0.9547603	best

In [ ]:
sub1 : 0.95546
sub2 : 0.95532
# depth=5 sub1
 : 0.95551
# depth=4 sub1
 : 0.95554

In [5]:
sub = pd.read_csv(SUB_PATH)

sub['Heart Disease'] = test_pred
sub.to_csv("../outputs/submission2.csv", index=False)